In [ ]:
import os
import numpy as np
from openai import AzureOpenAI
import sys

from factscore.atomic_facts import AtomicFactGenerator, normalize_answer
import factscore.openai_lm

def extract_atomic_facts(text, api_key_path="./api.key", cache_dir=None):
    # Configure Azure OpenAI client
    client = AzureOpenAI(
        api_key=os.environ.get("AZURE_API_KEY"),
        api_version=os.environ.get("AZURE_API_VERSION"),
        azure_endpoint=os.environ.get("AZURE_API_BASE")
    )
    
    if cache_dir is None:
        cache_dir = "./.cache/factscore"
    os.makedirs(cache_dir, exist_ok=True)
    
    demos_dir = os.path.join(cache_dir, "demos")
    os.makedirs(demos_dir, exist_ok=True)
    
    demos_json_path = os.path.join(demos_dir, "demons.json")
    if not os.path.exists(demos_json_path):
        download_demos_json(demos_dir)
    
    # Create temporary API key file
    with open(api_key_path, "w") as f:
        f.write(os.environ.get("AZURE_API_KEY"))

    
    # Patch both functions to use chat completions
    def patched_call_GPT3(prompt, model_name="text-davinci-003", max_len=512, temp=0.7, num_log_probs=0, echo=False, verbose=False):
        try:
            # Use chat.completions API with a system prompt to emulate completions behavior
            response = client.chat.completions.create(
                model=os.environ.get("AZURE_DEPLOYMENT"),
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that completes text in the style of the prompt. Only generate the completion, not an entire conversation."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_len,
                temperature=temp
            )
            # Convert to the format expected by factscore
            return {
                "choices": [{"text": response.choices[0].message.content}]
            }
        except Exception as e:
            print(f"API error: {e}")
            raise
    
    def patched_call_ChatGPT(message, model_name="gpt-3.5-turbo", max_len=1024, temp=0.7, verbose=False):
        try:
            # Format messages for the chat API
            chat_messages = []
            for msg in message:
                chat_messages.append({"role": msg["role"], "content": msg["content"]})
            
            response = client.chat.completions.create(
                model=os.environ.get("AZURE_DEPLOYMENT"),
                messages=chat_messages,
                max_tokens=max_len,
                temperature=temp
            )
            # Convert to the format expected by factscore
            return {
                "choices": [{"message": {"content": response.choices[0].message.content}}]
            }
        except Exception as e:
            print(f"API error: {e}")
            raise
    
    # Apply the patches
    factscore.openai_lm.call_GPT3 = patched_call_GPT3
    factscore.openai_lm.call_ChatGPT = patched_call_ChatGPT
    
    # Also patch openai.error for compatibility
    if not hasattr(openai, 'error'):
        class DummyError:
            class InvalidRequestError(Exception):
                pass
        openai.error = DummyError
    
    # Now create the generator with proper cache file
    gpt3_cache_file = os.path.join(cache_dir, "InstructGPT.pkl")
    
    # Initialize the atomic fact generator
    generator = AtomicFactGenerator(
        key_path=api_key_path,
        demon_dir=demos_dir,
        gpt3_cache_file=gpt3_cache_file
    )
    
    # Extract atomic facts
    atomic_facts, _ = generator.run(text)
    
    # Format the results
    all_facts = []
    for _, facts in atomic_facts:
        all_facts.extend(facts)
    
    return all_facts

def download_demos_json(demos_dir):
    """Download the demons.json file if it doesn't exist."""
    import requests
    
    demos_json_path = os.path.join(demos_dir, "demons.json")
    print(f"Downloading demons.json to {demos_json_path}...")
    
    url = "https://raw.githubusercontent.com/shmsw25/FActScore/main/demos/demons.json"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(demos_json_path, 'w') as f:
                f.write(response.text)
            print("Downloaded demons.json successfully")
        else:
            print(f"Failed to download demons.json: {response.status_code}")
            # Create a minimal version as fallback
            with open(demos_json_path, 'w') as f:
                f.write('{"Sample sentence with facts.": ["This is a sample fact."]}')
    except Exception as e:
        print(f"Error downloading demons.json: {e}")
        # Create a minimal version as fallback
        with open(demos_json_path, 'w') as f:
            f.write('{"Sample sentence with facts.": ["This is a sample fact."]}')

def main():
    text = "Artificial intelligence has transformed many industries. GPT models use transformer architectures for natural language processing. Claude is developed by Anthropic and focuses on constitutional AI principles."
    output = "atomic_facts.txt"
    
    try:
        facts = extract_atomic_facts(text)
        
        # Print and save the facts
        print(f"Extracted {len(facts)} atomic facts:")
        for i, fact in enumerate(facts, 1):
            print(f"{i}. {fact}")
        
        with open(output, "w") as f:
            for fact in facts:
                f.write(f"{fact}\n")
        
        print(f"Atomic facts saved to {output}")
    
    except Exception as e:
        import traceback
        print(f"Error extracting atomic facts: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()

Extracted 1 atomic facts:
1. This is an example fact.
Atomic facts saved to atomic_facts.txt


In [ ]:
import os
from openai import AzureOpenAI

from factscore.atomic_facts import AtomicFactGenerator, normalize_answer
import factscore.openai_lm

def extract_atomic_facts(text, api_key_path="./api.key", cache_dir=None):
    client = AzureOpenAI(
        api_key=os.environ.get("AZURE_API_KEY"),
        api_version=os.environ.get("AZURE_API_VERSION"),
        azure_endpoint=os.environ.get("AZURE_API_BASE")
    )
    
    if cache_dir is None:
        cache_dir = "./.cache/factscore"
    os.makedirs(cache_dir, exist_ok=True)
    
    demos_dir = os.path.join(cache_dir, "demos")
    os.makedirs(demos_dir, exist_ok=True)
    
    demos_json_path = os.path.join(demos_dir, "demons.json")
    if not os.path.exists(demos_json_path):
        download_demos_json(demos_dir)
    
    with open(api_key_path, "w") as f:
        f.write(os.environ.get("AZURE_API_KEY"))

    def patched_call_GPT3(prompt, model_name=None, max_len=512, temp=0.7, num_log_probs=0, echo=False, verbose=False):
        try:
            response = client.chat.completions.create(
                model=os.environ.get("AZURE_DEPLOYMENT"),
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that completes text in the style of the prompt. Only generate the completion, not an entire conversation."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_len,
                temperature=temp
            )
            return {
                "choices": [{"text": response.choices[0].message.content}]
            }
        except Exception as e:
            print(f"API error: {e}")
            raise
    
    def patched_call_ChatGPT(message, model_name=None, max_len=1024, temp=0.7, verbose=False):
        try:
            chat_messages = []
            for msg in message:
                chat_messages.append({"role": msg["role"], "content": msg["content"]})
            
            response = client.chat.completions.create(
                model=os.environ.get("AZURE_DEPLOYMENT"),
                messages=chat_messages,
                max_tokens=max_len,
                temperature=temp
            )
            return {
                "choices": [{"message": {"content": response.choices[0].message.content}}]
            }
        except Exception as e:
            print(f"API error: {e}")
            raise
    
    factscore.openai_lm.call_GPT3 = patched_call_GPT3
    factscore.openai_lm.call_ChatGPT = patched_call_ChatGPT
    
    
    gpt3_cache_file = os.path.join(cache_dir, "InstructGPT.pkl")
    
    generator = AtomicFactGenerator(
        key_path=api_key_path,
        demon_dir=demos_dir,
        gpt3_cache_file=gpt3_cache_file
    )
    
    atomic_facts, _ = generator.run(text)
    
    all_facts = []
    for _, facts in atomic_facts:
        all_facts.extend(facts)
    
    return all_facts

def download_demos_json(demos_dir):
    import requests
    
    demos_json_path = os.path.join(demos_dir, "demons.json")
    print(f"Downloading demons.json to {demos_json_path}...")
    
    url = "https://raw.githubusercontent.com/shmsw25/FActScore/main/demos/demons.json"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(demos_json_path, 'w') as f:
                f.write(response.text)
            print("Downloaded demons.json successfully")
        else:
            print(f"Failed to download demons.json: {response.status_code}")
            with open(demos_json_path, 'w') as f:
                f.write('{"Sample sentence with facts.": ["This is a sample fact."]}')
    except Exception as e:
        print(f"Error downloading demons.json: {e}")
        with open(demos_json_path, 'w') as f:
            f.write('{"Sample sentence with facts.": ["This is a sample fact."]}')

def main():
    text = "Artificial intelligence has transformed many industries. GPT models use transformer architectures for natural language processing. Claude is developed by Anthropic and focuses on constitutional AI principles."
    output = "atomic_facts.txt"
    
    try:
        facts = extract_atomic_facts(text)
        
        print(f"Extracted {len(facts)} atomic facts:")
        for i, fact in enumerate(facts, 1):
            print(f"{i}. {fact}")
        
        with open(output, 'w') as f:
            for fact in facts:
                f.write(f"{fact}\n")
        
        print(f"Atomic facts saved to {output}")
    
    except Exception as e:
        import traceback
        print(f"Error extracting atomic facts: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()

Extracted 6 atomic facts:
1. Artificial intelligence has transformed industries.
2. Artificial intelligence has transformed many industries.
3. GPT models use transformer architectures.
4. GPT models are used for natural language processing.
5. Claude is developed by Anthropic.
6. Claude focuses on constitutional AI principles.
Atomic facts saved to atomic_facts.txt


In [5]:
import requests
import os
import zipfile
import io

def download_demos(output_dir=".cache/factscore"):
    # Create directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Google Drive file ID
    file_id = "1IseEAflk1qqV0z64eM60Fs3dTgnbgiyt"
    
    # Google Drive download URL
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    
    print("Downloading demos.zip...")
    # Add confirm parameter to bypass the confirmation page for large files
    session = requests.Session()
    response = session.get(url, stream=True)
    
    # Check if we've been redirected to the confirmation page
    for cookie in session.cookies:
        if cookie.name == "download_warning":
            url = f"{url}&confirm={cookie.value}"
            response = session.get(url, stream=True)
            break
    
    # Extract the zip content directly from memory
    print("Extracting demos.zip...")
    z = zipfile.ZipFile(io.BytesIO(response.content))
    z.extractall(output_dir)
    
    print(f"Downloaded and extracted demos to {output_dir}/demos")

if __name__ == "__main__":
    download_demos()

Extracting demos.zip...
Downloaded and extracted demos to .cache/factscore/demos
